# Spot the emotion
## Under Threat of Scream
- Script to fit the psychometric and gaussian functions

In [1]:
# import relevant modules

import os
from warnings import simplefilter


import numpy as np
import pandas as pd

from scipy import stats

from lmfit.models import GaussianModel, StepModel
from lmfit import Parameters, Model
import pathlib
from itertools import product
import matplotlib.pyplot as plt
from scipy.stats import linregress

# suppress warning about fragmented dataframe. It's really not a problem...
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [2]:
# set data directory and get list of IDs
data_dir = pathlib.Path("..\\..\\STE_data\\")
f_names = list(data_dir.glob("*.csv"))

In [3]:
# have a look at the data
sub_df = pd.read_csv(f_names[0])
sub_df.query('Outcome_p_sad == 40 or Outcome_p_sad == 60').query('Expectedness == "UE"')

,Trial_N,Block_N,Contingency,Expectedness,Cue_idx,Cue_name,Outcome_idx,Outcome_name,Outcome_p_sad,Response_idx,Response_name,Response_RT,Confidence_idx,Confidence_name,Confidence_RT
30,31,1,0.75,UE,1,Low,0,Happy,40,1.0,Sad,1276.0,0.0,Low,706.0
51,52,1,0.75,UE,0,High,1,Sad,60,1.0,Sad,584.0,1.0,High,170.0
60,61,1,0.75,UE,1,Low,0,Happy,40,0.0,Happy,858.0,0.0,Low,115.0
76,77,1,0.75,UE,0,High,1,Sad,60,1.0,Sad,684.0,0.0,Low,199.0
79,80,1,0.75,UE,0,High,1,Sad,60,1.0,Sad,789.0,1.0,High,168.0
86,87,1,0.75,UE,1,Low,0,Happy,40,0.0,Happy,774.0,0.0,Low,125.0
89,90,1,0.75,UE,0,High,1,Sad,60,1.0,Sad,540.0,1.0,High,184.0
94,95,1,0.75,UE,1,Low,0,Happy,40,0.0,Happy,972.0,0.0,Low,122.0
105,106,2,0.25,UE,1,Low,1,Sad,60,1.0,Sad,530.0,1.0,High,132.0
112,113,2,0.25,UE,0,High,0,Happy,40,1.0,Sad,708.0,0.0,Low,125.0


In [4]:
# define functions to retrieve summary data from individual raw files


# set query terms to define subsets
query_terms = {'all': 'Trial_N < 1000',
               'expected': 'Expectedness == "E"',
               'unexpected': 'Expectedness == "UE"',
               'stable': 'Block_N == 1',
               'volatile': 'Block_N > 1'
              }

# define logistic function with upper and lower asymptotes
from numpy import exp
def logistic_model(x, mu, sigma, gamma, lambda_,):
    y = gamma + (1 - gamma - lambda_)*(1./(1+exp(-1*(sigma)*(x-mu))))
    return y



# get correct/incorrect
def get_accuracy(sub_df, cond):
    d = dict() # initialise output dictionary

    # split into Sad and Happy (stimuli)
    S_df = sub_df.query('Outcome_idx == 1')
    H_df = sub_df.query('Outcome_idx == 0')

    # Get proportions
    p_correct = sum(sub_df.Outcome_idx == sub_df.Response_idx) / len(sub_df);
    p_S_as_S = sum(S_df.Response_idx == 1) / len(S_df)
    p_H_as_H = sum(H_df.Response_idx == 0) / len(H_df)
    p_S_as_H = sum(S_df.Response_idx == 0) / len(S_df)
    p_H_as_S = sum(H_df.Response_idx == 1) / len(H_df)

    # add results to dict
    d[cond + '_p_correct'] = p_correct
    d[cond + '_sad_as_sad'] = p_S_as_S
    d[cond + '_happy_as_happy'] = p_H_as_H
    d[cond + '_sad_as_happy'] = p_S_as_H
    d[cond + '_happy_as_sad'] = p_H_as_S
    
    # return dict
    return d 

# get avg confidence (per condition)
def get_avg_confidence(sub_df, cond):
    d = dict() # initialise output dictionary

    # loop through subsets
    for q in query_terms.keys():
        subset_df = sub_df.query(query_terms[q])
        d[cond + '_' + q + '_avg_conf'] = subset_df.Confidence_idx.mean()
    return d


# get RT
def get_RT(sub_df, cond):
    d = dict() # initialise output dictionary

    # loop through subsets
    for q in query_terms.keys():
        subset_df = sub_df.query(query_terms[q])
        d[cond + '_' + q + '_respRT'] = subset_df.Response_RT.mean()
        d[cond + '_' + q + '_respLogRT'] = np.log(subset_df.Response_RT.mean())
        d[cond + '_' + q + '_confRT'] = subset_df.Confidence_RT.mean()
        d[cond + '_' + q + '_confLogRT'] = np.log(subset_df.Confidence_RT.mean())
    return d


# get RT (by noise level)
def get_RT_per_noise(sub_df, cond):
    d = dict()
    
    # Loop through subsets
    for q in query_terms.keys():
        subset_df = sub_df.query(query_terms[q])
        d[cond + '_' + q + '_noise1_respRT'] = subset_df.query('Outcome_p_sad == 0 or Outcome_p_sad == 100')['Response_RT'].mean()
        d[cond + '_' + q + '_noise2_respRT'] = subset_df.query('Outcome_p_sad == 20 or Outcome_p_sad == 80')['Response_RT'].mean()
        d[cond + '_' + q + '_noise3_respRT'] = subset_df.query('Outcome_p_sad == 40 or Outcome_p_sad == 60')['Response_RT'].mean()

        d[cond + '_' + q + '_noise1_respLogRT'] = np.log(subset_df.query('Outcome_p_sad == 0 or Outcome_p_sad == 100')['Response_RT'].mean())
        d[cond + '_' + q + '_noise2_respLogRT'] = np.log(subset_df.query('Outcome_p_sad == 20 or Outcome_p_sad == 80')['Response_RT'].mean())
        d[cond + '_' + q + '_noise3_respLogRT'] = np.log(subset_df.query('Outcome_p_sad == 40 or Outcome_p_sad == 60')['Response_RT'].mean())
    return d
    

# fit logistic curves to responses of subset of data (e.g. stable block)
def fit_responses(sub_df, cond, ID):

    # initialise output dictionaries
    d = dict()
    p = dict()
    
    # get x
    x = np.array([0,20,40,60,80,100])

    # loop through subsets
    for q in query_terms.keys():
        subset_df = sub_df.query(query_terms[q])
        y = [subset_df.query('Outcome_p_sad == @i').Response_idx.mean() for i in x]
        
        x = np.asarray(x, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64)
    
        # fit model
        #model = StepModel(form='logistic') # initialise
        model = Model(logistic_model)

        #model = StepModel(form='logistic')
        init_params = Parameters()
        init_params.add('mu', value = 50, min = .01, max = 99.99) # PSE (centre)
        init_params.add('sigma', value = 1, min = 0, max = 1) # Slope
        init_params.add('gamma', value = 0, min = 0, max = 0.5) # upper asymptote
        init_params.add('lambda_', value = 0, min = 0, max = 0.5) # lower asymptote
        
        fit_result = model.fit(y, init_params, x=x) # fit model to data
        params = fit_result.params # get parameters of fit model
        p[str(ID) + '_' + cond + '_' + q] = params
        #d[cond + '_' + q + '_pse'] = params['center'].value
        d[cond + '_' + q + '_pse'] = params['mu'].value
        d[cond + '_' + q + '_slope'] = params['sigma'].value
    return d, p


# fit confidence
def fit_confidence(sub_df, cond, ID):
    # initialise output dictionaries
    d = dict()
    p = dict()
    
    # get x
    x = np.array([0,20,40,60,80,100])

    # loop through subsets
    for q in query_terms.keys():
        subset_df = sub_df.query(query_terms[q])
        y = np.array([1 - subset_df.query('Outcome_p_sad == @i').Confidence_idx.mean() for i in x])
        
        x = np.asarray(x, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64)
    
        # fit model
        model = GaussianModel() # initialise
        init_params = model.guess(y, x=x) # initial guess for parameters
        fit_result = model.fit(y, init_params, x=x) # fit model to data
        params = fit_result.params # get parameters of fit model
        p[str(ID) + '_' + cond + '_' + q] = params
        d[cond + '_' + q + '_uncertainty_mu'] = params['center'].value
        d[cond + '_' + q + '_uncertainty_spread'] = params['fwhm'].value
    return d, p


In [5]:
# Preallocate dataframe
IDs = np.unique([f.name.split('_')[0] for f in f_names])
full_df = pd.DataFrame(index = IDs)
full_df.index = full_df.index.astype('int')

# Initialise dictionary for sub-specific paramers
sub_resp_params = dict()
sub_conf_params = dict()

In [6]:
# loop through files and add data to dataframe
for f in f_names:
    ID = int(f.name.split('_')[0])
    group = f.name.split('_')[1]
    cond = f.name.split('.')[0].split('_')[2]
    sub_df = pd.read_csv(f)

    # Add group to dataframe
    full_df.loc[ID, 'Group'] = group

    # remove missed responses
    n_missed = sum(sub_df.Response_idx.isna())
    full_df.loc[ID, cond + '_N_missed'] = n_missed
    
    if n_missed < 20: # approximately 10%, might have to play with this threshold

        # remove missing responses
        sub_df = sub_df.loc[~sub_df.Response_idx.isna()]
        
        # get accuracy
        d = get_accuracy(sub_df, cond)
        full_df.loc[ID, d.keys()] = d
    
        # get avg confidence
        d = get_avg_confidence(sub_df, cond)
        full_df.loc[ID, d.keys()] = d

        # get RT
        d = get_RT(sub_df, cond)
        full_df.loc[ID, d.keys()] = d

        # get RT per noise level
        d = get_RT_per_noise(sub_df, cond)
        full_df.loc[ID, d.keys()] = d
    
        # get parameters for psychometric function
        d, p = fit_responses(sub_df, cond, ID)
        full_df.loc[ID, d.keys()] = d
        sub_resp_params.update(p)
    
        # get parameters for gaussian function fit
        d, p = fit_confidence(sub_df, cond, ID)
        full_df.loc[ID, d.keys()] = d
        sub_conf_params.update(p)


In [7]:
# calculate effects of expectedness/stability
for m, c in product(['avg_conf', 'pse', 'slope', 'uncertainty_mu', 'uncertainty_spread', 'respRT', 'respLogRT', 'confRT'], ['Safe', 'Threat']):
    full_df[c + '_expectedness-effect_' + m] = full_df[c + '_expected_' + m] - full_df[c + '_unexpected_' + m]
    full_df[c + '_volatility-effect_' + m] = full_df[c + '_stable_' + m] - full_df[c + '_volatile_' + m]

In [8]:
full_df.head()

,Group,Safe_N_missed,Safe_p_correct,Safe_sad_as_sad,Safe_happy_as_happy,Safe_sad_as_happy,Safe_happy_as_sad,Safe_all_avg_conf,Safe_expected_avg_conf,Safe_unexpected_avg_conf,...,Threat_expectedness-effect_respRT,Threat_volatility-effect_respRT,Safe_expectedness-effect_respLogRT,Safe_volatility-effect_respLogRT,Threat_expectedness-effect_respLogRT,Threat_volatility-effect_respLogRT,Safe_expectedness-effect_confRT,Safe_volatility-effect_confRT,Threat_expectedness-effect_confRT,Threat_volatility-effect_confRT
10369536,A,1.0,0.837696,1.000000,0.677083,0.000000,0.322917,0.628272,0.657343,0.541667,...,-49.979167,-23.760417,0.014367,0.218114,-0.069017,-0.033377,2.659091,92.717105,11.743056,-9.385417
10369701,A,0.0,0.848958,0.927083,0.770833,0.072917,0.229167,0.734375,0.743056,0.708333,...,175.011111,-4.997917,-0.021643,0.037928,0.273368,-0.007265,44.038194,-29.419792,47.148611,82.408333
10369772,A,1.0,0.795812,0.884211,0.708333,0.115789,0.291667,0.984293,0.986014,0.979167,...,46.702083,42.211458,0.039085,0.177213,0.098694,0.087042,-56.456804,81.836886,-36.895139,36.107292
10369779,A,1.0,0.942408,0.979167,0.905263,0.020833,0.094737,0.947644,0.958042,0.916667,...,5.827083,-22.509375,-0.069133,0.316345,0.009555,-0.036824,20.525714,158.949441,23.963889,76.245833
10369781,A,0.0,0.880208,0.947917,0.812500,0.052083,0.187500,0.770833,0.770833,0.770833,...,-21.861111,-17.645833,0.058771,-0.019978,-0.030812,-0.025063,-19.756944,80.718750,1.125000,-0.312500


In [9]:
# save
full_df.to_csv('psychometric_function_fits.csv')

In [10]:
# Load transdiagnostic measures and add to dataframe
#transdiagnostic_df = pd.read_csv('..\\DATA\\predictedFactorScores.csv')
#transdiagnostic_df.set_index('subjIDs', drop=True, inplace=True)
#transdiagnostic_df = transdiagnostic_df.drop('Group', axis = 'columns')

#binarise (high/low for each trait)
#medians = transdiagnostic_df.median()
#for t in transdiagnostic_df.columns:
#    transdiagnostic_df[t + '_binary'] = transdiagnostic_df[t] > medians[t]

#full_df[['AD', 'Compul', 'SW', 'AD_binary', 'Compul_binary', 'SW_binary']] = transdiagnostic_df

In [11]:
# Load VAS and add to dataframe
#VAS_df = pd.read_csv('..\\DATA\\state_anxiety_VAS.csv')
#VAS_df.set_index('ID', drop=True, inplace=True)
#
#VAS_df['safe>baseline'] = VAS_df['pre_safe'] - VAS_df['expt_start']
#VAS_df['threat>baseline'] = VAS_df['pre_threat'] - VAS_df['expt_start']
#VAS_df['threat>safe'] = VAS_df['pre_threat'] - VAS_df['pre_safe']
#
#full_df[['VAS_' + c for c in VAS_df.columns]] = VAS_df

In [12]:
# Load attention check and add to dataframe
#attention_df = pd.read_csv('..\\DATA\\attention_check.csv');
#attention_df.set_index('ID', drop=True, inplace=True)
#attention_df = attention_df.drop('Group', axis = 'columns')
#attention_df = attention_df.rename(columns = 
#    {
#        'Safe_Check_1': 'Attention_check_safe1',
#        'Safe_Check_2': 'Attention_check_safe2',
#        'Safe_Check_3': 'Attention_check_safe3',
#        'Threat_Check_1': 'Attention_check_threat1',
#        'Threat_Check_2': 'Attention_check_threat2',
#        'Threat_Check_3': 'Attention_check_threat3',
#    })
#full_df[
#    ['Attention_check_safe1', 
#     'Attention_check_safe2', 
#     'Attention_check_safe3', 
#     'Attention_check_threat1', 
#     'Attention_check_threat2', 
#     'Attention_check_threat3']] = attention_df

In [13]:
# calculate inclusion/exclusion
#full_df['Attention_check_sum'] = full_df[[
#    'Attention_check_safe1', 
#    'Attention_check_safe2', 
#    'Attention_check_safe3', 
#    'Attention_check_threat1', 
#    'Attention_check_threat2', 
#    'Attention_check_threat3']].sum(axis=1, skipna=True)
#
#full_df['Attention_check_include'] = full_df['Attention_check_sum'] > 4 # i.e. missed no more than 2
#full_df['N_missing_safe_include'] = full_df['Safe_N_missed'] < 20
#full_df['N_missing_threat_include'] = full_df['Threat_N_missed'] < 20
#
#full_df['Include'] = full_df[
#    ['Attention_check_include', 
#     'N_missing_safe_include', 
#     'N_missing_threat_include']].sum(axis=1) == 3
#
#full_df['original'] = (full_df['Group'] == 'A') | (full_df['Group'] == 'B')

In [14]:
#full_df['Include'].sum()

In [15]:
# save
#full_df.to_csv('..\\fits.csv')